## Import

In [ ]:
import torch
import pandas as pd
import random
from transformers import EsmTokenizer, EsmForMaskedLM
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tqdm
import re
import os
import torch.nn.functional as F
from collections import Counter

## Random mutation

In [ ]:
# ご自身の抗体の野生型（WT）配列に書き換えてください
seq_name = "1mel"

wt_sequence = "VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREGVAAINMGGGITYYADSVKGRFTISQDNAKNTVYLLMNSLEPEDTAIYYCAADSTIYASYYECGHGLSTGGYGYDSWGQGTQVTVSS"

# 生成したい変異体配列の数
num_sequences_to_generate = 1000

# 変異数（複数変異体の場合）
num_mutations = 2  # 2変異、3変異などに変更可能

# 出力するCSVファイル名
output_dir = "outputs"
output_csv_filename = f"multi_mutations_random_{num_mutations}mut_{seq_name}_{num_sequences_to_generate}.csv"


In [ ]:
# パラメータ設定（パターン3bと同じパラメータを使用）
seq_name_3a = seq_name  # パターン3bと同じ設定を使用
num_sequences_to_generate_3a = num_sequences_to_generate
num_mutations_3a = num_mutations
output_csv_filename_3a = f"multi_mutations_random_{num_mutations_3a}mut_{seq_name_3a}_{num_sequences_to_generate_3a}.csv"

# Clean the input sequence
amino_acids_3a = "ACDEFGHIKLMNPQRSTVWY"
mutant_amino_acids_3a = "ADEFGHIKLMNPQRSTVWY"  # Cysを除く
wt_sequence_3a = "".join(re.findall(f"[{amino_acids_3a}]", wt_sequence.upper()))
print(f"Pattern 3a: Random strategy")
print(f"Cleaned WT sequence: {wt_sequence_3a[:30]}... (Length: {len(wt_sequence_3a)})")
print(f"Mutation target amino acids (excluding Cys): {mutant_amino_acids_3a}")

# --- 複数変異体生成（完全ランダム戦略） ---
print(f"\nGenerating {num_sequences_to_generate_3a} mutant sequences with {num_mutations_3a} mutations using random strategy...")

# Random seedを設定
random_seed_3a = 42  # シード値を変更可能
random.seed(random_seed_3a)
print(f"Random seed set to: {random_seed_3a}")

# 変異体情報を保存するリスト
all_mutation_data_3a = []
# 重複チェック用のセット
generated_sequences_3a = set()

start_time_3a = time.time()
last_reported_count_3a = 0

# 生成ループ
while len(all_mutation_data_3a) < num_sequences_to_generate_3a:
    # 1. ランダムに変異ポジションを選択（重複なし、num_mutations個）
    if num_mutations_3a > len(wt_sequence_3a):
        print(f"Error: num_mutations ({num_mutations_3a}) exceeds sequence length ({len(wt_sequence_3a)})")
        break
    
    mutation_positions_3a = random.sample(range(len(wt_sequence_3a)), num_mutations_3a)
    
    # 2. 各位置で、Cys以外のアミノ酸からランダムに選択
    mutations_applied_3a = []
    new_sequence_list_3a = list(wt_sequence_3a)
    
    for pos in mutation_positions_3a:
        original_aa = wt_sequence_3a[pos]
        # 元のアミノ酸以外のアミノ酸からランダムに選択
        available_aas = [aa for aa in mutant_amino_acids_3a if aa != original_aa]
        if len(available_aas) > 0:
            new_aa = random.choice(available_aas)
            mutations_applied_3a.append({
                "position": pos + 1,  # 1-based indexing
                "original_aa": original_aa,
                "mutated_aa": new_aa
            })
            new_sequence_list_3a[pos] = new_aa
    
    # 3. 変異数をチェック（完全一致のみ）
    if len(mutations_applied_3a) == num_mutations_3a:
        new_mutated_sequence_3a = "".join(new_sequence_list_3a)
        
        # 重複チェック
        if new_mutated_sequence_3a not in generated_sequences_3a:
            generated_sequences_3a.add(new_mutated_sequence_3a)
            all_mutation_data_3a.append({
                "sequence": new_mutated_sequence_3a,
                "mutations": mutations_applied_3a,
                "num_mutations": len(mutations_applied_3a)
            })
            
            # 進捗表示
            current_count_3a = len(all_mutation_data_3a)
            if current_count_3a > last_reported_count_3a and current_count_3a % 10 == 0:
                elapsed_time_3a = time.time() - start_time_3a
                print(f"Generated {current_count_3a} / {num_sequences_to_generate_3a} sequences... ({elapsed_time_3a:.2f} seconds)")
                last_reported_count_3a = current_count_3a

print(f"\nSequence generation complete.")
total_time_3a = time.time() - start_time_3a
print(f"Total time taken: {total_time_3a:.2f} seconds")
print(f"Generated {len(all_mutation_data_3a)} unique sequences with exactly {num_mutations_3a} mutations.")

# --- DataFrame作成とCSV出力 ---
import json

print(f"\nCreating DataFrame and saving to CSV...")

# DataFrame用のデータリストを作成
df_data_3a = []
for mutation_data in all_mutation_data_3a:
    row_data = {
        "sequence": mutation_data["sequence"],
        "num_mutations": mutation_data["num_mutations"],
        "mutations": json.dumps(mutation_data["mutations"]),  # JSON文字列に変換
    }
    df_data_3a.append(row_data)

# DataFrameを作成
df_output_3a = pd.DataFrame(df_data_3a)

# 出力ディレクトリを作成
os.makedirs(output_dir, exist_ok=True)

# CSVファイルに保存
output_path_3a = os.path.join(output_dir, output_csv_filename_3a)
df_output_3a.to_csv(output_path_3a, index=False)

print(f"Successfully saved {len(df_output_3a)} sequences to '{output_path_3a}'")
print(f"\nColumns: {list(df_output_3a.columns)}")
print(f"\nFirst few rows:")
print(df_output_3a.head())


Pattern 3a: Random strategy
Cleaned WT sequence: VQLQASGGGSVQAGGSLRLSCAASGYTIGP... (Length: 132)
Mutation target amino acids (excluding Cys): ADEFGHIKLMNPQRSTVWY

Generating 1000 mutant sequences with 2 mutations using random strategy...
Random seed set to: 42
Generated 10 / 1000 sequences... (0.00 seconds)
Generated 20 / 1000 sequences... (0.00 seconds)
Generated 30 / 1000 sequences... (0.00 seconds)
Generated 40 / 1000 sequences... (0.00 seconds)
Generated 50 / 1000 sequences... (0.00 seconds)
Generated 60 / 1000 sequences... (0.00 seconds)
Generated 70 / 1000 sequences... (0.00 seconds)
Generated 80 / 1000 sequences... (0.00 seconds)
Generated 90 / 1000 sequences... (0.00 seconds)
Generated 100 / 1000 sequences... (0.00 seconds)
Generated 110 / 1000 sequences... (0.00 seconds)
Generated 120 / 1000 sequences... (0.00 seconds)
Generated 130 / 1000 sequences... (0.00 seconds)
Generated 140 / 1000 sequences... (0.00 seconds)
Generated 150 / 1000 sequences... (0.00 seconds)
Generated 160

## pLM mutation

In [ ]:
# ご自身の抗体の野生型（WT）配列に書き換えてください
seq_name = "1mel"

wt_sequence = "VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREGVAAINMGGGITYYADSVKGRFTISQDNAKNTVYLLMNSLEPEDTAIYYCAADSTIYASYYECGHGLSTGGYGYDSWGQGTQVTVSS"

# 使用するESM-2モデル
model_name = "facebook/esm2_t33_650M_UR50D"

# --- モデルとトークナイザーのロード ---
print(f"Loading ESM-2 model: {model_name}...")
try:
    tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name)
    model.eval() # モデルを評価モード（推論モード）に設定
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# --- 実行例 ---
# パラメータ設定
seq_name_3c = seq_name
num_variants_3c = 100000
mutation_counts_3c = [2]  # 変異数を指定

# PLL計算を実行するかどうか
calculate_pll_3c = True  # TrueにするとPLL計算を実行（計算時間がかかります）

# PLL上位X%のみをCSVに出力するかどうか
filter_top_pll_3c = True  # TrueにするとPLL上位X%のみをCSVに出力
top_pll_percent_3c = 1  # 上位何%を出力するか（0.1-100.0の範囲）

# 出力ファイル名（フィルタリング処理後に更新される可能性あり）
output_csv_filename_3c_base = f"esm2_650M_large_scale_variants_{seq_name_3c}_{num_variants_3c}"

print(f"WT sequence: {wt_sequence[:30]}... (Length: {len(wt_sequence)})")
print(f"Target variants: {num_variants_3c}")
print(f"Mutation counts: {mutation_counts_3c}\\n")

Loading ESM-2 model: facebook/esm2_t33_650M_UR50D...
Model and tokenizer loaded successfully.


In [ ]:
# Check if a GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model.to(device)
model.eval()

def generate_large_scale_variants(wt_seq, num_variants=10000, mutation_counts=[2, 3, 4, 5]):
    """
    ESM-2の確率分布（Zero-shot）に基づいて、高速に大量の変異体を生成する
    """
    print(f"Processing WT sequence (Length: {len(wt_seq)})...")
    
    # 1. モデルによる確率行列の計算（これだけGPUで行う）
    inputs = tokenizer(wt_seq, return_tensors="pt", add_special_tokens=False).to(device)
    input_ids = inputs["input_ids"]  # shape: [1, seq_len]
    seq_len = input_ids.shape[1]
    
    with torch.no_grad():
        logits = model(input_ids).logits  # [1, seq_len, vocab_size]
        probs = F.softmax(logits, dim=-1)[0]  # [seq_len, vocab_size]
    
    # 2. 確率行列の整理（CPUで処理）
    probs = probs.cpu().numpy()
    wt_ids = input_ids[0].cpu().numpy()
    
    # 変異候補のリスト作成
    mutation_candidates = []
    weights = []
    
    valid_aa_indices = []
    # 特殊トークンを除外したアミノ酸インデックスを取得
    for i in range(tokenizer.vocab_size):
        char = tokenizer.decode([i])
        if char not in ["<cls>", "<pad>", "<eos>", "<unk>", "-", "X", "B", "U", "Z", "O"]:
            valid_aa_indices.append(i)
            
    print("Building mutation probability map...")
    
    for pos in range(seq_len):
        wt_aa_id = wt_ids[pos]
        
        for aa_id in valid_aa_indices:
            if aa_id == wt_aa_id:
                continue
            
            mut_prob = probs[pos, aa_id]
            
            if mut_prob < 0.001: 
                continue

            # (位置, 変異アミノ酸文字)
            mutation_candidates.append((pos, tokenizer.decode([aa_id])))
            
            # 重みとして確率を使用
            weights.append(mut_prob)
            
    # 重みの正規化
    weights = np.array(weights)
    weights = weights / weights.sum()
    
    print(f"Total valid single mutations found: {len(mutation_candidates)}")
    print(f"Generating {num_variants} variants...")
    
    generated_variants = set()
    results = []
    
    # 3. 高速サンプリング
    while len(results) < num_variants:
        # 変異数Nをランダムに決定
        n_muts = random.choice(mutation_counts)
        
        # 重みに従ってN個の変異を選ぶ
        chosen_indices = np.random.choice(len(mutation_candidates), size=n_muts*2, replace=False, p=weights)
        
        current_mutations = []
        used_positions = set()
        
        for idx in chosen_indices:
            pos, aa = mutation_candidates[idx]
            if pos not in used_positions:
                current_mutations.append((pos, aa))
                used_positions.add(pos)
            
            if len(current_mutations) == n_muts:
                break
        
        if len(current_mutations) < n_muts:
            continue
            
        # 順番を揃えて識別子にする（重複排除のため）
        current_mutations.sort(key=lambda x: x[0])
        variant_id = tuple(current_mutations)
        
        if variant_id in generated_variants:
            continue
            
        generated_variants.add(variant_id)
        
        # 配列生成
        seq_list = list(wt_seq)
        mut_info_list = []
        for pos, aa in current_mutations:
            wt_aa = wt_seq[pos]
            seq_list[pos] = aa
            mut_info_list.append({
                "position": pos + 1,  # 1-based indexing
                "original_aa": wt_aa,
                "mutated_aa": aa
            })
            
        results.append({
            "sequence": "".join(seq_list),
            "mutations": mut_info_list,
            "num_mutations": n_muts
        })
        
        if len(results) % 1000 == 0:
            print(f"Generated {len(results)} variants...")

    return results

# 変異体生成
variants_3c = generate_large_scale_variants(wt_sequence, num_variants=num_variants_3c, mutation_counts=mutation_counts_3c)

print(f"\\nGeneration complete. Generated {len(variants_3c)} variants.\\n")

# 結果の確認（最初の5件）
print("--- Example Generated Variants ---")
for v in variants_3c[:5]:
    muts_str = ", ".join([f"{m['original_aa']}{m['position']}{m['mutated_aa']}" for m in v['mutations']])
    print(f"[{v['num_mutations']} muts] {muts_str}")

# 変異が導入された位置を確認
print("\n--- Mutation Position Analysis ---")
used_positions = set()
for variant in variants_3c:
    for m in variant["mutations"]:
        used_positions.add(m["position"] - 1)  # 0-based

all_positions = set(range(len(wt_sequence)))
unused_positions = sorted(all_positions - used_positions)

print(f"Total sequence length: {len(wt_sequence)}")
print(f"Positions with mutations: {len(used_positions)}")
print(f"Positions without mutations: {len(unused_positions)}")
if unused_positions:
    # 1-based indexで表示
    unused_positions_1based = [pos + 1 for pos in unused_positions]
    print(f"Unused positions (1-based): {unused_positions_1based}")
    print(f"Unused positions (0-based): {unused_positions}")

# --- PLL（擬似対数尤度）計算（バッチ処理版、オプション） ---
def calc_pseudo_pll_batch_3c(variants, model, tokenizer, wt_sequence):
    """
    変異体リストに対してバッチ処理でPLLを計算
    - 各変異位置の確率分布をキャッシュして効率化
    - 各変異体のPLLを計算
    
    Args:
        variants (list): 変異体のリスト（各要素は{"sequence": str, "mutations": list, ...}）
        model: ESM-2モデル
        tokenizer: ESM-2トークナイザ
        wt_sequence (str): 野生型配列
        
    Returns:
        pll_values (list): 各変異体のPLL値のリスト
    """
    model.eval()
    
    # 1. 全変異体から使用されている変異位置を収集
    all_mutation_positions = set()
    for variant in variants:
        mutations = variant["mutations"]
        for m in mutations:
            all_mutation_positions.add(m["position"] - 1)  # 1-based to 0-based
    
    all_mutation_positions = sorted(all_mutation_positions)
    print(f"Computing log probabilities for {len(all_mutation_positions)} unique mutation positions...")
    
    # 2. 各変異位置について確率分布を事前計算（キャッシュ）
    position_log_probs_cache = {}
    
    with torch.no_grad():
        for pos in tqdm.tqdm(all_mutation_positions, desc="Precomputing log probs"):
            # 各変異位置をマスク
            sequence_list = list(wt_sequence)
            sequence_list[pos] = tokenizer.mask_token
            masked_sequence = "".join(sequence_list)
            
            # トークナイズ
            inputs = tokenizer(masked_sequence, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # マスク位置のインデックスを取得
            masked_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1][0]
            
            # モデルで予測
            outputs = model(**inputs)
            logits = outputs.logits[0, masked_index, :]
            log_probs = F.log_softmax(logits, dim=-1)
            
            # CPUに移動してキャッシュ
            position_log_probs_cache[pos] = log_probs.cpu()
    
    # 3. 各変異体のPLLを計算（キャッシュを使用）
    print(f"Calculating PLL for {len(variants)} variants...")
    pll_values = []
    
    for variant in tqdm.tqdm(variants, desc="Calculating PLL"):
        mutations = variant["mutations"]
        
        if len(mutations) == 0:
            pll_values.append(float("-inf"))
            continue
        
        total_log_likelihood = 0.0
        for m in mutations:
            pos = m["position"] - 1  # 1-based to 0-based
            mutated_aa = m["mutated_aa"]
            mutated_token_id = tokenizer.convert_tokens_to_ids(mutated_aa)
            
            # キャッシュから確率分布を取得
            log_probs = position_log_probs_cache[pos]
            total_log_likelihood += log_probs[mutated_token_id].item()
        
        # PLL = 変異箇所の平均対数尤度
        pll = total_log_likelihood / len(mutations)
        pll_values.append(pll)
    
    return pll_values

# フィルタリング後の配列リストに対してPLLを計算（オプション）
if calculate_pll_3c:
    print(f"\\nCalculating PLL (mutations only) for {len(variants_3c)} variants using batch processing...")
    pll_start_time = time.time()
    
    # バッチ処理でPLLを計算
    pll_values_3c = calc_pseudo_pll_batch_3c(variants_3c, model, tokenizer, wt_sequence)
    
    # 各変異体にPLL値を追加
    for i, variant in enumerate(variants_3c):
        variant["pll"] = pll_values_3c[i]
    
    pll_time = time.time() - pll_start_time
    print(f"PLL calculation finished in {pll_time:.2f} seconds.")
else:
    print(f"\\nPLL calculation skipped (calculate_pll_3c = False)")

# DataFrame作成とCSV出力
import json

df_data_3c = []
for variant in variants_3c:
    row_data = {
        "sequence": variant["sequence"],
        "num_mutations": variant["num_mutations"],
        "mutations": json.dumps(variant["mutations"]),  # JSON文字列に変換
    }
    # PLL列を条件付きで追加
    if calculate_pll_3c:
        row_data["pll"] = variant["pll"]
    df_data_3c.append(row_data)

df_output_3c = pd.DataFrame(df_data_3c)

# PLL上位X%でフィルタリング（オプション）
if filter_top_pll_3c and calculate_pll_3c:
    if "pll" not in df_output_3c.columns:
        print("\\nWarning: PLL column not found. Skipping filtering.")
    else:
        # PLL値でソート（降順：高い順）
        df_output_3c = df_output_3c.sort_values("pll", ascending=False)
        
        # 上位X%のインデックスを計算
        num_top = max(1, int(len(df_output_3c) * (top_pll_percent_3c / 100.0)))
        df_output_3c = df_output_3c.head(num_top).copy()
        
        print(f"\\nFiltered to top {top_pll_percent_3c}% (PLL): {len(df_output_3c)} variants")
        if len(df_output_3c) > 0:
            print(f"PLL range: {df_output_3c['pll'].min():.4f} to {df_output_3c['pll'].max():.4f}")
elif filter_top_pll_3c and not calculate_pll_3c:
    print("\\nWarning: filter_top_pll_3c is True but calculate_pll_3c is False. Skipping filtering.")

# 出力ファイル名を決定（フィルタリング処理に応じて更新）
if filter_top_pll_3c and calculate_pll_3c and "pll" in df_output_3c.columns:
    # フィルタリングが実行された場合、ファイル名に情報を追加
    output_csv_filename_3c = f"{output_csv_filename_3c_base}_top{top_pll_percent_3c}pct.csv"
else:
    # フィルタリングが実行されなかった場合
    output_csv_filename_3c = f"{output_csv_filename_3c_base}.csv"

# 出力ディレクトリを作成
os.makedirs(output_dir, exist_ok=True)

# CSVファイルに保存
output_path_3c = os.path.join(output_dir, output_csv_filename_3c)
df_output_3c.to_csv(output_path_3c, index=False)

print(f"\\nSuccessfully saved {len(df_output_3c)} sequences to '{output_path_3c}'")
print(f"\\nColumns: {list(df_output_3c.columns)}")
print(f"\\nFirst few rows:")
print(df_output_3c.head())

Using device: cuda
\n=== パターン3c: 高速大規模変異体生成 ===\n
WT sequence: VQLQASGGGSVQAGGSLRLSCAASGYTIGP... (Length: 132)
Target variants: 100000
Mutation counts: [2]\n
Processing WT sequence (Length: 132)...
Building mutation probability map...
Total valid single mutations found: 1205
Generating 100000 variants...
Generated 1000 variants...
Generated 2000 variants...
Generated 3000 variants...
Generated 4000 variants...
Generated 5000 variants...
Generated 6000 variants...
Generated 7000 variants...
Generated 8000 variants...
Generated 9000 variants...
Generated 10000 variants...
Generated 11000 variants...
Generated 12000 variants...
Generated 13000 variants...
Generated 14000 variants...
Generated 15000 variants...
Generated 16000 variants...
Generated 17000 variants...
Generated 18000 variants...
Generated 19000 variants...
Generated 20000 variants...
Generated 21000 variants...
Generated 22000 variants...
Generated 23000 variants...
Generated 24000 variants...
Generated 25000 variants...
Gen

Precomputing log probs: 100%|██████████| 124/124 [00:04<00:00, 30.23it/s]


Calculating PLL for 100000 variants...


Calculating PLL: 100%|██████████| 100000/100000 [00:00<00:00, 183199.46it/s]


PLL calculation finished in 4.71 seconds.
\nFiltered to top 1% (PLL): 1000 variants
PLL range: -0.9491 to -0.0521
\nSuccessfully saved 1000 sequences to 'outputs/esm2_650M_large_scale_variants_1mel_100000_top1pct.csv'
\nColumns: ['sequence', 'num_mutations', 'mutations', 'pll']
\nFirst few rows:
                                               sequence  num_mutations  \
118   VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
351   VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
1022  VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREG...              2   
525   VQLQASGGGSVQAGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREW...              2   
5529  VQLQASGGGSVQPGGSLRLSCAASGYTIGPYCMGWFRQAPGKEREG...              2   

                                              mutations       pll  
118   [{"position": 13, "original_aa": "A", "mutated... -0.052113  
351   [{"position": 46, "original_aa": "G", "mutated... -0.081810  
1022  [{"position": 13, "original_aa": "A", "mutated..